# 风矢量平均与非线性聚合

月平均u、v经常被用于构造风速代理：

$$
ws_{vec}=\sqrt{\bar u^2+\bar v^2}.
$$

这个量描述平均风矢量的模长。“先计算每个时刻的标量风速，再做时间平均”对应另一种统计量。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n = 30 * 24
speed = np.full(n, 8.0)

theta = np.linspace(0, 2 * np.pi, n, endpoint=False)
u = speed * np.cos(theta)
v = speed * np.sin(theta)

mean_scalar_speed = np.mean(np.sqrt(u**2 + v**2))
vector_mean_speed = np.sqrt(np.mean(u)**2 + np.mean(v)**2)

print(f"mean of scalar speed = {mean_scalar_speed:.3f} m/s")
print(f"magnitude of mean vector = {vector_mean_speed:.6f} m/s")

方向变化会在u、v平均时发生抵消。即使每个时刻的标量风速都保持8 m/s，平均风矢量也可能接近零。

实际月平均资料通常不会出现如此整齐的完整旋转，但方向变化带来的矢量抵消仍然存在。

In [ ]:
hours = np.arange(168)

plt.figure(figsize=(9, 4.5))
plt.plot(hours, u[:168], label="u")
plt.plot(hours, v[:168], label="v")
plt.xlabel("Hour")
plt.ylabel("Wind component (m/s)")
plt.legend()
plt.tight_layout()
plt.show()

## 空间平均与功率映射的顺序

假设一个区域内不同格点具有不同风速，可以比较两种处理：

$$
\mathrm{route\ A}=\frac{1}{N}\sum_i P(V_i),
$$

$$
\mathrm{route\ B}=P\left(\frac{1}{N}\sum_i V_i\right).
$$

只要P是非线性的，两条路线一般不会得到相同结果。

In [ ]:
def cf_proxy(ws):
    ws = np.asarray(ws, dtype=float)
    out = np.zeros_like(ws)
    partial = (ws >= 3.0) & (ws < 12.0)
    rated = (ws >= 12.0) & (ws <= 25.0)
    out[partial] = ((ws[partial] - 3.0) / 9.0) ** 3
    out[rated] = 1.0
    return out

ws_grid = np.array([
    [2.0, 4.0, 6.0, 8.0],
    [3.0, 5.0, 9.0, 12.0],
    [4.0, 7.0, 10.0, 14.0],
])

route_a = cf_proxy(ws_grid).mean()
route_b = float(cf_proxy(np.array([ws_grid.mean()]))[0])

print(f"mean wind speed = {ws_grid.mean():.3f} m/s")
print(f"mean[CF(ws)] = {route_a:.4f}")
print(f"CF(mean[ws]) = {route_b:.4f}")

In [ ]:
labels = ["mean[CF(ws)]", "CF(mean[ws])"]
values = [route_a, route_b]

plt.figure(figsize=(6, 4.5))
plt.bar(labels, values)
plt.ylabel("CF proxy")
plt.tight_layout()
plt.show()

## 与仓库案例的关系

CanESM5—ERA5案例使用月平均近地层u、v。案例中的ws和CF均作为代理量使用，这个限定来自变量定义和聚合顺序。

参考：

- ECMWF关于u、v到风速的定义说明  
  https://confluence.ecmwf.int/spaces/CKB/pages/133262398/
- Drobinski (2026), https://doi.org/10.1038/s44168-025-00332-4